# Importing Libraries

In [1]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd
from geopy.distance import geodesic

# Importing Dataframe

In [2]:
api_data = pd.read_csv('../01. Data/01. Raw Data/api_hotels_paris_rome_base_clean.csv')

In [3]:
api_data.shape

(2144, 15)

# Cleaning Dataframe

In [4]:
api_data.head()

,chain_code,iata_code,dupe_id,hotel_name,hotel_id,last_update,latitude,longitude,country_code,postal_code,city_name,address_lines,is_sponsored,master_chain_code,search_city
0,IW,PAR,700041562,MAISON ALBAR-LE CHAMPS-ELYSEES,IWPARMMP,2025-12-22 11:57:46,48.87541,2.29453,FR,75017,PARIS,['3 AVENUE MAC MAHON'],True,NaN,Paris
1,BN,PAR,700031222,OCCIDENTAL PARIS LEVALLOIS,BNPAR058,2026-04-09 11:21:08,48.89818,2.28087,FR,92300,PARIS,"['8 PLACE GEORGES POMPIDOU', 'LEVALLOIS PERRET']",True,NaN,Paris
2,WA,PAR,700027367,WLDRF ASTORIA VERSAILLES TRIANON PALACE,WAPARTRI,2026-04-11 16:24:20,48.81067,2.12022,FR,78000,VERSAILLES - PARIS,['1 BOULEVARD DE LA REINE'],True,EH,Paris
3,PN,PAR,700222607,THE PENINSULA PARIS,PNPARTPP,2026-03-06 08:51:54,48.87073,2.29340,FR,75116,PARIS,['19 AVENUE KLEBER'],True,NaN,Paris
4,HL,PAR,700024440,HILTON PARIS DEFENSE,HLPAR100,2026-04-14 08:25:09,48.89305,2.23857,FR,92053,LA DEFENSE,"['2 PLACE DE LA DEFENSE', 'CNIT-B']",True,EH,Paris


In [5]:
# checking for null values
api_data.isnull().sum()

chain_code              0
iata_code               0
dupe_id                 0
hotel_name              0
hotel_id                0
last_update             0
latitude                0
longitude               0
country_code            0
postal_code             1
city_name               0
address_lines           0
is_sponsored            0
master_chain_code    1239
search_city             0
dtype: int64

In [6]:
# dropping unecessary columns
api_data.drop(columns=['chain_code', 'iata_code', 'postal_code', 'is_sponsored', 'master_chain_code'], inplace=True)

In [7]:
# checking the dropping of columns
api_data.head()

,dupe_id,hotel_name,hotel_id,last_update,latitude,longitude,country_code,city_name,address_lines,search_city
0,700041562,MAISON ALBAR-LE CHAMPS-ELYSEES,IWPARMMP,2025-12-22 11:57:46,48.87541,2.29453,FR,PARIS,['3 AVENUE MAC MAHON'],Paris
1,700031222,OCCIDENTAL PARIS LEVALLOIS,BNPAR058,2026-04-09 11:21:08,48.89818,2.28087,FR,PARIS,"['8 PLACE GEORGES POMPIDOU', 'LEVALLOIS PERRET']",Paris
2,700027367,WLDRF ASTORIA VERSAILLES TRIANON PALACE,WAPARTRI,2026-04-11 16:24:20,48.81067,2.12022,FR,VERSAILLES - PARIS,['1 BOULEVARD DE LA REINE'],Paris
3,700222607,THE PENINSULA PARIS,PNPARTPP,2026-03-06 08:51:54,48.87073,2.29340,FR,PARIS,['19 AVENUE KLEBER'],Paris
4,700024440,HILTON PARIS DEFENSE,HLPAR100,2026-04-14 08:25:09,48.89305,2.23857,FR,LA DEFENSE,"['2 PLACE DE LA DEFENSE', 'CNIT-B']",Paris


In [8]:
# checking for duplicates
api_data.duplicated().sum()

np.int64(0)

In [9]:
# dropping more columns after duplicates check
api_data.drop(columns=['dupe_id', 'hotel_id', 'country_code', 'last_update'], inplace=True)

In [10]:
# checking the dropping of columns
api_data.head()

,hotel_name,latitude,longitude,city_name,address_lines,search_city
0,MAISON ALBAR-LE CHAMPS-ELYSEES,48.87541,2.29453,PARIS,['3 AVENUE MAC MAHON'],Paris
1,OCCIDENTAL PARIS LEVALLOIS,48.89818,2.28087,PARIS,"['8 PLACE GEORGES POMPIDOU', 'LEVALLOIS PERRET']",Paris
2,WLDRF ASTORIA VERSAILLES TRIANON PALACE,48.81067,2.12022,VERSAILLES - PARIS,['1 BOULEVARD DE LA REINE'],Paris
3,THE PENINSULA PARIS,48.87073,2.29340,PARIS,['19 AVENUE KLEBER'],Paris
4,HILTON PARIS DEFENSE,48.89305,2.23857,LA DEFENSE,"['2 PLACE DE LA DEFENSE', 'CNIT-B']",Paris


In [11]:
# definition of a function to check the unique values in each column

def unique_counts_col(df):
    for col in df.columns:
        print(f'Column: {col}')
        print(df[col].value_counts())
        print('-----------------------------')

unique_counts_col(api_data)

Column: hotel_name
hotel_name
HOTEL ALBERT 1ER               2
HOTEL MADISON                  2
IBIS STYLES PARIS NATION       2
HOTEL LORD BYRON               2
MERCURE ROMA CENTRO ST JOHN    2
                              ..
KOLBE HOTEL ROME               1
PALAZZO DELLE PIETRE           1
25ROOMS                        1
MARTA GUEST HOUSE              1
HOTEL MEDITERRANEO             1
Name: count, Length: 2139, dtype: int64
-----------------------------
Column: latitude
latitude
48.86735    4
48.87177    3
48.87679    3
48.81583    3
48.87137    3
           ..
41.86714    1
41.90726    1
41.89005    1
41.90050    1
41.90585    1
Name: count, Length: 1956, dtype: int64
-----------------------------
Column: longitude
longitude
2.30758     3
2.35042     3
2.28347     3
12.49730    3
2.28087     2
           ..
12.48402    1
12.47534    1
12.51628    1
12.46792    1
12.49895    1
Name: count, Length: 2016, dtype: int64
-----------------------------
Column: city_name
city_name
PARIS  

In [12]:
# dropping rows with 'Rome' in search_city column
api_data = api_data[api_data['search_city'] != 'Rome']

In [13]:
# checking the drop of rows
api_data['search_city'].value_counts()

search_city
Paris    1679
Name: count, dtype: int64

In [14]:
# Checking unique values for the 'city_name' column
api_data['city_name'].unique()

<StringArray>
[                'PARIS',    'VERSAILLES - PARIS',            'LA DEFENSE',
      'ROISSY EN FRANCE',              'COLOMBES',   'ISSY-LES-MOULINEAUX',
        'NOISY LE GRAND',       'CERNAY LA VILLE',                 'Paris',
               'CLAMART',
 ...
               'Gonesse',         'FONTAINEBLEAU',          'Rocquencourt',
      'PLESSIS-ROBINSON',          'LA COURNEUVE',   'Champigny sur Marne',
 'BAILLY-ROMAINVILLIERS',  'SAINT-OUEN-SUR-SEINE',           'LES MUREAUX',
       'MEUNG SUR LOIRE']
Length: 261, dtype: str

In [15]:
# standardize PARIS and Paris in city_name column
api_data['city_name'] = api_data['city_name'].str.strip().str.title()

In [16]:
# checking standarization of city_name column
api_data['city_name'].unique()

<StringArray>
[                'Paris',    'Versailles - Paris',            'La Defense',
      'Roissy En France',              'Colombes',   'Issy-Les-Moulineaux',
        'Noisy Le Grand',       'Cernay La Ville',               'Clamart',
                'Sevres',
 ...
               'Mouroux',         'Fontainebleau',          'Rocquencourt',
      'Plessis-Robinson',          'La Courneuve',   'Champigny Sur Marne',
 'Bailly-Romainvilliers',  'Saint-Ouen-Sur-Seine',           'Les Mureaux',
       'Meung Sur Loire']
Length: 255, dtype: str

In [17]:
# dropping hotels that are further than 50km from Paris city center
paris_center = (48.8566, 2.3522)
max_distance_km = 50

def is_within_paris_area(row):
    coords = (row['latitude'], row['longitude'])
    return geodesic(paris_center, coords).km <= max_distance_km

api_data = api_data[api_data.apply(is_within_paris_area, axis=1)]

In [18]:
# checking the dropping of rows
api_data['city_name'].unique()

<StringArray>
[                'Paris',    'Versailles - Paris',            'La Defense',
      'Roissy En France',              'Colombes',   'Issy-Les-Moulineaux',
        'Noisy Le Grand',       'Cernay La Ville',               'Clamart',
                'Sevres',
 ...
                 'Bondy',     'St-Ouen L Aum One',    'Saint Ouen Laumone',
          'Rocquencourt',      'Plessis-Robinson',          'La Courneuve',
   'Champigny Sur Marne', 'Bailly-Romainvilliers',  'Saint-Ouen-Sur-Seine',
           'Les Mureaux']
Length: 240, dtype: str

In [19]:
# checking for duplicates after dropping columns and rows
api_data.duplicated().sum()

np.int64(0)

In [20]:
# checking data types of the columns
api_data.dtypes

hotel_name           str
latitude         float64
longitude        float64
city_name            str
address_lines        str
search_city          str
dtype: object

In [21]:
# standardize all string columns in the dataframe
str_cols = api_data.select_dtypes(include='str').columns

api_data[str_cols] = api_data[str_cols].apply(lambda x: x.str.strip().str.title())

In [22]:
api_data.head()

,hotel_name,latitude,longitude,city_name,address_lines,search_city
0,Maison Albar-Le Champs-Elysees,48.87541,2.29453,Paris,['3 Avenue Mac Mahon'],Paris
1,Occidental Paris Levallois,48.89818,2.28087,Paris,"['8 Place Georges Pompidou', 'Levallois Perret']",Paris
2,Wldrf Astoria Versailles Trianon Palace,48.81067,2.12022,Versailles - Paris,['1 Boulevard De La Reine'],Paris
3,The Peninsula Paris,48.87073,2.29340,Paris,['19 Avenue Kleber'],Paris
4,Hilton Paris Defense,48.89305,2.23857,La Defense,"['2 Place De La Defense', 'Cnit-B']",Paris


In [23]:
# fixing the address_lines column to be more readable
api_data['address_lines'] = api_data['address_lines'].str.replace(r"[\[\]']", "", regex=True).str.strip()

In [24]:
# checking if standardization worked for address_lines column   
api_data.head()

,hotel_name,latitude,longitude,city_name,address_lines,search_city
0,Maison Albar-Le Champs-Elysees,48.87541,2.29453,Paris,3 Avenue Mac Mahon,Paris
1,Occidental Paris Levallois,48.89818,2.28087,Paris,"8 Place Georges Pompidou, Levallois Perret",Paris
2,Wldrf Astoria Versailles Trianon Palace,48.81067,2.12022,Versailles - Paris,1 Boulevard De La Reine,Paris
3,The Peninsula Paris,48.87073,2.29340,Paris,19 Avenue Kleber,Paris
4,Hilton Paris Defense,48.89305,2.23857,La Defense,"2 Place De La Defense, Cnit-B",Paris


In [ ]:
# exporting cleaned data to CSV file
api_data.to_pickle('../01. Data/02. Clean Data/api_hotels_paris_clean.pkl')